# Preamble

In [1]:
import os
import cv2 as cv
import numpy as np
from PIL import Image
import pandas as pd
from torchvision.transforms import transforms
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder
from PIL import Image
from torch import nn
import torch.optim as optim
import matplotlib.pyplot as plt
import time

%matplotlib inline

# Preparing Data for Training

In [ ]:
image_path = [[],[]]
labels = [[],[]]

# Folder produced by src/augment.py — contains train/ and val/ subfolders,
# each with one subfolder per class.
input_path = 'data/train_val_Split_Output'

j = 0

for dataset in os.listdir(input_path):
    for label in os.listdir(os.path.join(input_path,dataset)):
        for image in os.listdir(os.path.join(input_path,dataset,label)):
            image_path[j].append(os.path.join(input_path,dataset,label,image))
            labels[j].append(label)
    j += 1

In [3]:
train_df = pd.DataFrame({'image_path':image_path[0],'label':labels[0]})
# test_df = pd.DataFrame({'image_path':image_path[0],'label':labels[0]})
val_df = pd.DataFrame({'image_path':image_path[1],'label':labels[1]})

full_frame = pd.concat([train_df,val_df])

print(full_frame['label'].unique())

# LabelEncoder() object fits labels with an interger from 0 to n_labels-1
label_encoder = LabelEncoder()
label_encoder.fit(full_frame['label'])

['KEK_LAPIS' 'KUIH_KASWI_PANDAN' 'KUIH_KETAYAP' 'KUIH_LAPIS'
 'KUIH_SERI_MUKA' 'KUIH_TALAM' 'KUIH_UBI_KAYU' 'ONDE_ONDE']


LabelEncoder()

In [4]:
train_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.ConvertImageDtype(torch.float32),
    transforms.RandomErasing(p=0.3),
    transforms.ColorJitter(brightness=0.3),
    # this is the mean and standard deviation for [R,G,B] of the imagenet dataset
    # we can do better normalisation by using the mean and std dev of our
    # dataset
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize((224,224)),
    transforms.ConvertImageDtype(torch.float32),
    # this is the mean and standard deviation for [R,G,B] of the imagenet dataset
    # we can do better normalisation by using the mean and std dev of our
    # dataset
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

In [5]:
class CustomImageDataset(Dataset):
  def __init__(self, dataframe, transform = None):
    # transform = None initialises default transform as None
    self.dataframe = dataframe
    self.transform = transform
    # transform text labels into numerical labels
    # Keep labels on CPU (do NOT use .to(device) here)
    self.labels = torch.tensor(label_encoder.transform(dataframe["label"]))

  def __getitem__(self, index):
    # iloc[a,b] returns index at [row = a, column = b]
    # note that axis 0 refers to rows and axis 1 refers to columns
    row = self.dataframe.iloc[index]
    img_path = row["image_path"]
    label = self.labels[index]
    image = Image.open(img_path).convert('RGB')
    if self.transform:
      # Keep labels on CPU (do NOT use .to(device) here)
      image = self.transform(image)
    #both image and label on cpu
    return image, label

  def __len__(self):
    #returns how many rows you have in dataframe
    return len(self.dataframe)

In [6]:
train_dataset = CustomImageDataset(train_df, train_transform)
# test_dataset = CustomImageDataset(test_df, transform)
val_dataset = CustomImageDataset(val_df, val_transform)

In [7]:
# # learning rate
# LR = 1e-4
# # batch size
# BATCH_SIZE = 32
# # number of epochs
# EPOCHS = 10
# # SGD momentum
# momentum = 0.9

# Model

In [8]:
from torchvision import models
from torchvision.models import ResNet50_Weights
from hyperopt import fmin, tpe, hp, Trials, STATUS_OK
from torch.optim.lr_scheduler import ReduceLROnPlateau

In [9]:
# resnet = models.resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
#  #Example to print the names of each layer block
# for name, layer in resnet.named_children():
#     print(f'{name}')

In [10]:
# class CustomResNet(nn.Module):
#     def __init__(self):
#         super(CustomResNet, self).__init__()
#         self.resnet = models.resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)

# #        Freeze ALL RESNET Layers
# #        for param in self.resnet.parameters():
# #            param.requires_grad = False
        
# #        Freeze layers up to `layer3`
#         for name, param in self.resnet.named_parameters():
#             if "layer3" not in name and "layer4" not in name:
#                 param.requires_grad = False  # Persistent setting
        
#         self.resnet.fc = nn.Identity()
#         self.classifier = nn.Sequential(
#             nn.Flatten(),
#             nn.Linear(2048, 1024),
#             nn.ReLU(),
#             nn.Dropout(p=dropout),
#             nn.Linear(1024, 512),
#             nn.ReLU(),
#             nn.Dropout(p=dropout),
#             nn.Linear(512, 8)
#          )
#     def forward(self, x):
#         x = self.resnet(x)
#         x = self.classifier(x)
#         return x

In [11]:
# device = "cuda" if torch.cuda.is_available() else "cpu"
# model = CustomResNet().to(device)
# from torchsummary import summary
# summary(model, input_size = (3, 224, 224))

# Training

In [12]:
space = {
    'learning_rate': hp.loguniform('learning_rate', -10, -4),  # ~ 5e-5 -> 2e-2
    'batch_size': hp.choice('batch_size', [16, 32, 64, 128]),
    'dropout': hp.uniform('dropout',0.1,0.6),
    'weight_decay': hp.loguniform('weight_decay', -8, -3), # ~ 3e-4 -> 5e-2
    'momentum' : hp.uniform('momentum',0.90,0.99)
}

In [13]:
def objective(params):

    LR = params['learning_rate']
    batch_size = params['batch_size']
    dropout = params['dropout']
    WEIGHT_DECAY = params['weight_decay']
    momentum = params['momentum']
    

    #
    # ========================= MODEL =========================
    #

    class CustomResNet(nn.Module):
        def __init__(self):
            super(CustomResNet, self).__init__()
            self.resnet = models.resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
            for name, param in self.resnet.named_parameters():
                if "layer3" not in name and "layer4" not in name:
                    param.requires_grad = False 
            
            self.resnet.fc = nn.Identity()
            self.classifier = nn.Sequential(
                nn.Flatten(),
                nn.Linear(2048, 1024),
                nn.ReLU(),
                nn.Dropout(p=dropout),
                nn.Linear(1024, 512),
                nn.ReLU(),
                nn.Dropout(p=dropout),
                nn.Linear(512, 8)
             )
        def forward(self, x):
            x = self.resnet(x)
            x = self.classifier(x)
            return x

    #
    # ========================= MODEL =========================
    #


    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=True)

    device = "cuda" if torch.cuda.is_available() else "cpu" 

    model = CustomResNet().to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(
        model.parameters(), 
        lr=LR,
        momentum=momentum,
        weight_decay = WEIGHT_DECAY
    )
    scheduler = ReduceLROnPlateau(
        optimizer,
        mode = 'min',
        factor = 0.5,
        patience =  2,
    )

    # initialises the best validation loss as pos. infinity
    best_combined_loss = float('inf')

    for epoch in range(5):

        epoch_train_loss = 0.0
        
        model.train()
        
        for inputs_train, labels_train in train_loader:
            inputs_train, labels_train = inputs_train.to(device), labels_train.to(device)
    
            for param in model.parameters():
              param.grad = None
            
            outputs_train = model(inputs_train)
            loss_train = criterion(outputs_train, labels_train)
            loss_train.backward()
            optimizer.step()

            epoch_train_loss += loss_train.item() * inputs_train.size(0)

        avg_train_loss = epoch_train_loss / len(train_dataset)
        
        model.eval()

        epoch_val_loss = 0.0
        
        with torch.no_grad():
            for inputs_val, labels_val in val_loader:
                inputs_val, labels_val = inputs_val.to(device), labels_val.to(device)
                
                outputs_val = model(inputs_val)
                epoch_val_loss += criterion(outputs_val, labels_val).item()*inputs_val.size(0)

        avg_val_loss = epoch_val_loss / len(val_dataset)

        # Calculate combined loss
        generalization_gap = avg_val_loss - avg_train_loss
        combined_loss = avg_val_loss + 0.5 * generalization_gap

        scheduler.step(avg_val_loss)

        if combined_loss < best_combined_loss:
            best_combined_loss = combined_loss

    return {'loss': best_combined_loss, 'status': STATUS_OK}

In [14]:
trials = Trials()
best = fmin(
    fn=objective,
    space=space,
    algo=tpe.suggest,
    max_evals=50,  # Number of trials
    trials=trials
)

print(best)

100%|███████████████████████████████████████████| 50/50 [4:21:14<00:00, 313.48s/trial, best loss: -0.38704191512531705]
{'batch_size': np.int64(3), 'dropout': np.float64(0.4164126193234352), 'learning_rate': np.float64(0.005668124403105755), 'momentum': np.float64(0.9349709064491925), 'weight_decay': np.float64(0.008656136981267964)}


In [15]:
from hyperopt import space_eval
actual_best_params = space_eval(space, best)
print(actual_best_params)

{'batch_size': 128, 'dropout': 0.4164126193234352, 'learning_rate': 0.005668124403105755, 'momentum': 0.9349709064491925, 'weight_decay': 0.008656136981267964}


In [ ]:
import pickle
# Save the trials object so notebook 6 (BO analysis) can load and plot it.
pickle.dump(trials, open("results/final_bo.pkl", "wb"))

In [17]:
print(trials.best_trial['result']['loss'])

-0.38704191512531705


# Test & Analysis

In [18]:
# fig, axs = plt.subplots(nrows=1,ncols=2,figsize=(15,5))

# axs[0].plot(batch_loss_train_plot,label="Training Loss")
# axs[0].set_title("Training Loss over Batches")
# axs[0].set_xlabel("Batchs")
# axs[0].set_ylabel("Loss")

# axs[1].plot(batch_acc_train_plot,label="Training Accuracy")
# axs[1].set_title("Training Accuracy over Batches")
# axs[1].set_xlabel("Batch")
# axs[1].set_ylabel("Accuracy")

# #plt.savefig('batch_v5 (RESNET).png')

# plt.show()

In [19]:
# fig, axs = plt.subplots(nrows=1,ncols=2,figsize=(15,5))

# axs[0].plot(epoch_loss_train_plot,label="Training Loss")
# axs[0].plot(epoch_loss_val_plot,label="Validation Loss")
# axs[0].legend()
# axs[0].set_title("Training and Validation Loss over Epochs")
# axs[0].set_xlabel("Epochs")
# axs[0].set_ylabel("Loss")

# axs[1].plot(epoch_acc_train_plot,label="Training Accuracy")
# axs[1].plot(epoch_acc_val_plot,label="Validation Accuracy")
# axs[1].legend()
# axs[1].set_title("Training and Validation Accuracy over Epochs")
# axs[1].set_xlabel("Epochs")
# axs[1].set_ylabel("Accuracy")

# #plt.savefig('epoch_v5 (RESNET).png')

# plt.show()

# Save & Export

In [20]:
#print("Model's state_dict:")
#for param_tensor in model.state_dict():
#    print(param_tensor, '\t', model.state_dict()[param_tensor].size())

In [21]:
# Print optimizer's state_dict
#print("Optimizer's state_dict:")
#for var_name in optimizer.state_dict():
#    print(var_name, "\t", optimizer.state_dict()[var_name])

In [22]:
#torch.save(model.state_dict(), 'model(RESNETPartialFrozen).pt')

In [23]:
#torch.save(model,'entire_model.pt')

In [ ]:
# (Removed a leftover `torch.save(model.state_dict(), '')` cell that errored on an
#  empty filename. The BO notebook only searches hyperparameters; use notebook 4
#  to train and save the final model with the best params found above.)